# Synthetic Data Augmentation with Conditional Flow Matching

## Part 2

In this section, we evaluate the performance of the trained baseline classifier on the Fashion MNIST test set, using top-1 classification accuracy (proportion of samples for which the predicted class $-$ i.e., the class with the highest logit $-$ matches the ground truth label) and the macro-averaged (unweighted mean across all classes) $\mathsf F_1$ score.

## Setup

In [1]:
!find . -mindepth 1 -exec rm -rf {} + &> /dev/null
!git clone https://github.com/ZhangLyndon/FlowMatchingAugmentation . > /dev/null 2>&1

In [2]:
!pip install -qU -r requirements.txt

In [3]:
import os
import argparse

# Reduce CUDA memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torchvision
import numpy as np

# Components for initializing an ImageNet-pretrained ResNet-18 classifier, fine-
# tuning it on Fashion MNIST, and evaluating classification performance on base-
# line, low-data, and synthetically augmented settings.
from classification import (ClassificationTrainer,
                            create_classifier, ResNetClassifier,
                            SyntheticDataGenerator, SyntheticAugmentationEvaluator,
                            create_augmented_dataset)

# Utilities for loading the Fashion MNIST dataset, computing top-k categorical
# accuracy and average cross-entropy loss, and saving training results.
from utils import get_dataloaders, AverageMeter, accuracy, save_results

import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams["font.family"] = "DejaVu Sans Mono"

In [4]:
# Configure augmentation evaluation pipeline
augmentation_args = argparse.Namespace(data_root = "./data",
                                       batch_size = 16,
                                       num_workers = 0,
                                       epochs = 25,
                                       lr = 0.001,
                                       weight_decay = 1e-4,
                                       step_size = 15,
                                       gamma = 0.1,
                                       synthetic_data_dir = "./images",
                                       real_ratio = 1.0,
                                       classification_dir = "./results/classification",
                                       augmentation_dir = "./results/augmentation",
                                       checkpoint_dir = "./checkpoints",
                                       save_interval = 20,
                                       seed = 42)

# Create directory to store synthetic augmentation results
os.makedirs(augmentation_args.augmentation_dir, exist_ok = True)

# Set random seed for reproducibility
torch.manual_seed(augmentation_args.seed)
np.random.seed(augmentation_args.seed)

# Obtain the DataLoader for the test set
_, test_loader = get_dataloaders(root_dir = augmentation_args.data_root,
                                 batch_size = augmentation_args.batch_size,
                                 num_workers = augmentation_args.num_workers)

# Set the guidance scale to 1.0. This dummy value represents the base guided
# vector field without additional reinforcement.
guidance_scale = 1.0

# Evaluate the performance of the trained baseline classifier on the test set,
# and report the top-1 classification accuracy and macro-averaged F1 score. The
# macro F1 score helps reveal systematic underperformance on individual classes.
evaluator = SyntheticAugmentationEvaluator(augmentation_args, guidance_scale)
model_path = os.path.join(augmentation_args.checkpoint_dir, "resnet_best_val_loss.pt")
evaluator.evaluate_baseline(model_path, test_loader)

100%|██████████| 26.4M/26.4M [00:02<00:00, 12.7MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 200kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.75MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 10.7MB/s]


Baseline Evaluation: {'accuracy': 92.19, 'f1_macro': 0.921823418912893, 'total_samples': 10000}


We observe that the baseline classifier achieves a top-1 accuracy (proportion of samples for which the predicted class matches the ground truth) of 92.19% and a similar, macro-averaged $\mathsf F_1$ score (unweighted mean across all classes) of 0.9218, indicating fairly uniform performance across classes.